# Comment Topic Clustering — Dataproc/GCP-ready version

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

import matplotlib.pyplot as plt
from functools import reduce


In [ ]:
# =========================
# CONFIG: แก้ cell นี้ก่อนรันบน Dataproc/GCP
# =========================

DATASET_NAME = "comment"

# สำหรับรัน local/sample ใช้ชื่อไฟล์เดิมได้
# สำหรับ Dataproc/GCP ให้เปลี่ยนเป็น GCS path เช่น:
INPUT_PATH = "gs://reddit-ai-2/process_data/Comment_Simple_or_non_tag/comment_sentimented.parquet"
# INPUT_PATH = "process_data_Sampling_Topic_sampling_comment.parquet"

# สำหรับ Dataproc/GCP แนะนำให้เปลี่ยนเป็น gs://...
OUTPUT_BASE_PATH = "gs://reddit-ai-2/process_data/Comment_Clustering"
MODEL_OUTPUT_PATH = "gs://reddit-ai-2/process_data/Comment_Clustering/model"
# OUTPUT_BASE_PATH = f"clustering_output/{DATASET_NAME}"
# MODEL_OUTPUT_PATH = f"kmeans_model/{DATASET_NAME}"

SAVE_OUTPUTS = True

# ปรับได้ตามผล Elbow/Silhouette + business interpretation
OPTIMAL_K = 5
MIN_K = 2
MAX_K = 10

# ไม่ใส่ sentiment เข้า feature clustering เพื่อลด leakage
FEATURE_TOPICS = [
    "usecase_code",
    "usecase_write",
    "usecase_data",
    "usecase_creative",
    "usecase_research",
    "prod_affordable",
    "prod_usability",
    "prod_performance",
    "user_churn",
]

SENTIMENT_TOPICS = ["sentiment_positive", "sentiment_negative"]

print("DATASET_NAME:", DATASET_NAME)
print("INPUT_PATH:", INPUT_PATH)
print("OUTPUT_BASE_PATH:", OUTPUT_BASE_PATH)
print("MODEL_OUTPUT_PATH:", MODEL_OUTPUT_PATH)


In [ ]:
spark = SparkSession.builder.appName(f"{DATASET_NAME}_Topic_Clustering").getOrCreate()

df = spark.read.parquet(INPUT_PATH)

print(f"จำนวนแถวทั้งหมด: {df.count():,} แถว")
print(f"จำนวนคอลัมน์ทั้งหมด: {len(df.columns):,} คอลัมน์")
df.printSchema()
df.show(5, truncate=False)


In [ ]:
# =========================
# Data validation + basic cleaning
# =========================

required_columns = ["detected_topics"]
missing_required_columns = [c for c in required_columns if c not in df.columns]

if missing_required_columns:
    raise ValueError(f"Missing required columns: {missing_required_columns}")

# ถ้า detected_topics เป็น null ให้แทนเป็น array ว่าง
df_clean = df.withColumn(
    "detected_topics",
    F.coalesce(F.col("detected_topics"), F.array().cast("array<string>"))
)

print("จำนวน detected_topics ต่อแถว:")
df_clean.groupBy(F.size("detected_topics").alias("num_detected_topics")) \
    .count() \
    .orderBy("num_detected_topics") \
    .show(50, truncate=False)


In [ ]:
# =========================
# Feature engineering
# =========================
# สร้าง multi-hot topic features จาก detected_topics
# หมายเหตุ: ไม่ใช้ sentiment_positive / sentiment_negative ใน KMeans

df_processed = df_clean

for topic in FEATURE_TOPICS:
    df_processed = df_processed.withColumn(
        topic,
        F.when(F.array_contains(F.col("detected_topics"), topic), F.lit(1.0)).otherwise(F.lit(0.0))
    )

feature_topic_count_expr = reduce(lambda a, b: a + b, [F.col(topic) for topic in FEATURE_TOPICS])
df_processed = df_processed.withColumn("feature_topic_count", feature_topic_count_expr)

# แยก rows ที่ไม่มี non-sentiment topic ออก ไม่ให้เข้า KMeans
# เช่น detected_topics = [] หรือมีเฉพาะ sentiment_positive/sentiment_negative
df_feature_ready = df_processed.filter(F.col("feature_topic_count") > 0)
df_no_feature = df_processed.filter(F.col("feature_topic_count") == 0)

assembler = VectorAssembler(
    inputCols=FEATURE_TOPICS,
    outputCol="topic_features"
)

df_features = assembler.transform(df_feature_ready).cache()

total_rows = df_processed.count()
train_rows = df_features.count()
no_feature_rows = df_no_feature.count()

print(f"จำนวนแถวทั้งหมด: {total_rows:,}")
print(f"จำนวนแถวที่ใช้ train KMeans: {train_rows:,}")
print(f"จำนวนแถวที่แยกเป็น no_non_sentiment_topic: {no_feature_rows:,}")

if train_rows < 2:
    raise ValueError("Not enough rows with non-sentiment topic features for KMeans.")

df_features.select("detected_topics", "feature_topic_count", "topic_features", *FEATURE_TOPICS).show(10, truncate=False)


In [ ]:
# =========================
# Find candidate K using WSSSE + Silhouette
# =========================

max_k_allowed = min(MAX_K, train_rows - 1)

if max_k_allowed < MIN_K:
    raise ValueError(f"Not enough training rows to test K from {MIN_K} to {MAX_K}. train_rows={train_rows}")

evaluator = ClusteringEvaluator(
    featuresCol="topic_features",
    predictionCol="cluster_prediction",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

metrics = []

for k in range(MIN_K, max_k_allowed + 1):
    kmeans = KMeans(
        featuresCol="topic_features",
        k=k,
        seed=42,
        predictionCol="cluster_prediction"
    )
    model_k = kmeans.fit(df_features)
    predictions_k = model_k.transform(df_features)

    wssse = float(model_k.summary.trainingCost)
    silhouette = float(evaluator.evaluate(predictions_k))
    metrics.append((k, wssse, silhouette))

metrics_df = spark.createDataFrame(metrics, ["k", "wssse", "silhouette"]).orderBy("k")

print("K selection metrics:")
metrics_df.show(truncate=False)

# ใช้ pandas เฉพาะตาราง metrics ขนาดเล็กเพื่อ plot
metrics_pdf = metrics_df.toPandas()

plt.figure(figsize=(10, 5))
plt.plot(metrics_pdf["k"], metrics_pdf["wssse"], marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Within Set Sum of Squared Errors (WSSSE)")
plt.title("Elbow Method for Optimal K")
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(metrics_pdf["k"], metrics_pdf["silhouette"], marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Average Silhouette Score")
plt.title("Silhouette Score for Optimal K")
plt.grid(True)
plt.show()


In [ ]:
# =========================
# Train final KMeans model
# =========================

final_k = min(OPTIMAL_K, max_k_allowed)

if final_k != OPTIMAL_K:
    print(f"OPTIMAL_K={OPTIMAL_K} มากกว่าจำนวนที่ใช้ได้จริง จึงปรับเป็น final_k={final_k}")

kmeans = KMeans(
    featuresCol="topic_features",
    k=final_k,
    seed=42,
    predictionCol="cluster_label"
)

model = kmeans.fit(df_features)
df_clustered = model.transform(df_features)

original_cols = df_clean.columns

# เก็บเฉพาะคอลัมน์สำคัญสำหรับผลลัพธ์ ไม่เก็บ vector เพื่อให้ output เบากว่า
df_clustered_final = (
    df_clustered
    .withColumn("cluster_reason", F.lit("kmeans_topic_cluster"))
    .select(*original_cols, "feature_topic_count", "cluster_label", "cluster_reason")
)

df_no_feature_final = (
    df_no_feature
    .withColumn("cluster_label", F.lit(-1))
    .withColumn("cluster_reason", F.lit("no_non_sentiment_topic"))
    .select(*original_cols, "feature_topic_count", "cluster_label", "cluster_reason")
)

df_result = df_clustered_final.unionByName(df_no_feature_final).cache()

print(f"จำนวนแถวหลังรวมผลลัพธ์: {df_result.count():,}")
df_result.groupBy("cluster_label", "cluster_reason").count().orderBy("cluster_label").show(truncate=False)
df_result.show(10, truncate=False)


In [ ]:
# =========================
# Spark-native EDA: cluster summary
# =========================

agg_exprs = [
    F.count("*").alias("count_rows"),
    F.mean("feature_topic_count").alias("avg_feature_topic_count"),
]

if "ups" in df_result.columns:
    agg_exprs.append(F.mean("ups").alias("avg_upvotes"))

cluster_summary = (
    df_result
    .groupBy("cluster_label", "cluster_reason")
    .agg(*agg_exprs)
    .orderBy("cluster_label")
)

print("Cluster summary:")
cluster_summary.show(truncate=False)

# plot summary ขนาดเล็ก
cluster_summary_pdf = cluster_summary.toPandas()

plt.figure(figsize=(10, 5))
plt.bar(cluster_summary_pdf["cluster_label"].astype(str), cluster_summary_pdf["count_rows"])
plt.xlabel("Cluster Label")
plt.ylabel("Rows")
plt.title("Cluster Size")
plt.grid(axis="y")
plt.show()


In [ ]:
# =========================
# Spark-native EDA: topic proportions by cluster
# =========================

df_topics = (
    df_result
    .select("cluster_label", F.explode_outer("detected_topics").alias("topic"))
    .filter(F.col("topic").isNotNull())
    .filter(~F.col("topic").isin(SENTIMENT_TOPICS))
)

topic_counts = df_topics.groupBy("cluster_label", "topic").count()

topic_totals = (
    topic_counts
    .groupBy("cluster_label")
    .agg(F.sum("count").alias("topic_total"))
)

topic_proportion = (
    topic_counts
    .join(topic_totals, on="cluster_label", how="left")
    .withColumn("proportion", F.col("count") / F.col("topic_total"))
    .orderBy("cluster_label", "topic")
)

print("Topic proportion by cluster:")
topic_proportion.show(100, truncate=False)

# แปลงเฉพาะ summary table เล็ก ๆ เพื่อ plot
topic_pivot_pdf = (
    topic_proportion
    .groupBy("cluster_label")
    .pivot("topic")
    .agg(F.first("proportion"))
    .fillna(0)
    .orderBy("cluster_label")
    .toPandas()
)

if not topic_pivot_pdf.empty:
    topic_pivot_pdf = topic_pivot_pdf.set_index("cluster_label")
    topic_pivot_pdf.plot(kind="bar", stacked=True, figsize=(14, 7))
    plt.title("Proportion of Topics by Cluster")
    plt.xlabel("Cluster Label")
    plt.ylabel("Proportion")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("No non-sentiment topics available for topic proportion plot.")


In [ ]:
# =========================
# Spark-native EDA: sentiment proportions by cluster
# =========================

if "prediction" in df_result.columns:
    sentiment_counts = (
        df_result
        .filter(F.col("prediction").isNotNull())
        .groupBy("cluster_label", "prediction")
        .count()
    )

    sentiment_totals = (
        sentiment_counts
        .groupBy("cluster_label")
        .agg(F.sum("count").alias("sentiment_total"))
    )

    sentiment_proportion = (
        sentiment_counts
        .join(sentiment_totals, on="cluster_label", how="left")
        .withColumn("proportion", F.col("count") / F.col("sentiment_total"))
        .orderBy("cluster_label", "prediction")
    )

    print("Sentiment proportion by cluster:")
    sentiment_proportion.show(100, truncate=False)

    sentiment_pivot_pdf = (
        sentiment_proportion
        .groupBy("cluster_label")
        .pivot("prediction")
        .agg(F.first("proportion"))
        .fillna(0)
        .orderBy("cluster_label")
        .toPandas()
    )

    if not sentiment_pivot_pdf.empty:
        sentiment_pivot_pdf = sentiment_pivot_pdf.set_index("cluster_label")
        sentiment_pivot_pdf.plot(kind="bar", stacked=True, figsize=(12, 6))
        plt.title("Proportion of Sentiment by Cluster")
        plt.xlabel("Cluster Label")
        plt.ylabel("Proportion")
        plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        plt.show()
else:
    print("Column 'prediction' not found. Skip sentiment analysis.")


In [ ]:
# =========================
# Spark-native EDA: AI product proportions by cluster
# =========================

if "AI_Name" in df_result.columns:
    product_counts = (
        df_result
        .filter(F.col("AI_Name").isNotNull())
        .groupBy("cluster_label", "AI_Name")
        .count()
    )

    product_totals = (
        product_counts
        .groupBy("cluster_label")
        .agg(F.sum("count").alias("product_total"))
    )

    product_proportion = (
        product_counts
        .join(product_totals, on="cluster_label", how="left")
        .withColumn("proportion", F.col("count") / F.col("product_total"))
        .orderBy("cluster_label", "AI_Name")
    )

    print("Product count by cluster:")
    product_counts.orderBy("cluster_label", "AI_Name").show(100, truncate=False)

    print("Product proportion by cluster:")
    product_proportion.show(100, truncate=False)

    product_pivot_pdf = (
        product_proportion
        .groupBy("cluster_label")
        .pivot("AI_Name")
        .agg(F.first("proportion"))
        .fillna(0)
        .orderBy("cluster_label")
        .toPandas()
    )

    if not product_pivot_pdf.empty:
        product_pivot_pdf = product_pivot_pdf.set_index("cluster_label")
        product_pivot_pdf.plot(kind="bar", stacked=True, figsize=(12, 6))
        plt.title("Proportion of AI Product by Cluster")
        plt.xlabel("Cluster Label")
        plt.ylabel("Proportion")
        plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        plt.show()
else:
    print("Column 'AI_Name' not found. Skip product analysis.")


In [ ]:
# =========================
# Save outputs
# =========================

if SAVE_OUTPUTS:
    clustered_data_path = f"{OUTPUT_BASE_PATH}/clustered_data"
    metrics_path = f"{OUTPUT_BASE_PATH}/k_metrics"
    cluster_summary_path = f"{OUTPUT_BASE_PATH}/cluster_summary"
    topic_proportion_path = f"{OUTPUT_BASE_PATH}/topic_proportion"

    df_result.write.mode("overwrite").parquet(clustered_data_path)
    metrics_df.write.mode("overwrite").parquet(metrics_path)
    cluster_summary.write.mode("overwrite").parquet(cluster_summary_path)
    topic_proportion.write.mode("overwrite").parquet(topic_proportion_path)
    model.write().overwrite().save(MODEL_OUTPUT_PATH)

    print("Saved clustered data to:", clustered_data_path)
    print("Saved K metrics to:", metrics_path)
    print("Saved cluster summary to:", cluster_summary_path)
    print("Saved topic proportion to:", topic_proportion_path)
    print("Saved KMeans model to:", MODEL_OUTPUT_PATH)
else:
    print("SAVE_OUTPUTS=False, skip writing outputs.")


## Notes for interpretation

- `cluster_label = -1` หมายถึงแถวที่ไม่มี non-sentiment topic เพียงพอสำหรับ KMeans เช่น `detected_topics = []` หรือมีเฉพาะ sentiment topic
- sentiment ไม่ได้ถูกใช้เป็น feature ใน KMeans แล้ว ดังนั้นการวิเคราะห์ sentiment หลัง clustering จะตีความได้ clean กว่าเดิม
- ถ้าต้องการให้ sentiment เป็นส่วนหนึ่งของ cluster จริง ๆ ให้เพิ่ม `sentiment_positive` และ `sentiment_negative` กลับเข้า `FEATURE_TOPICS` แต่ต้องอธิบายในการ report ว่า cluster ถูกนิยามจาก sentiment ด้วย
- ก่อน deploy จริงควรเปลี่ยน `INPUT_PATH`, `OUTPUT_BASE_PATH`, และ `MODEL_OUTPUT_PATH` เป็น `gs://...`
